<a href="https://colab.research.google.com/github/h3692/company-buying-decision/blob/main/k_means_focus_100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

sns.set_theme(style="whitegrid")
df_unmodified = pd.read_csv("pipeline_results.csv")
df_unmodified.head()

In [ ]:
df_unmodified.info()

In [ ]:
df_compressed = pd.read_csv("pipeline_results.csv", usecols=["Company Name [canonical_name]", "Industry [industry]", "Employee Count [employee_count]", "Revenue [revenue]", "HQ State / Region [hq_region]", "Year Founded [year_founded]", "Ownership Type [public_private]", "Number of Locations [number_of_locations]", "Total Tech Tools [total_tech_count]", "Tech Complexity Score [tech_stack_complexity_score]", "Infrastructure Maturity Score [infrastructure_maturity]", "Endpoint Density [endpoint_density_proxy]", "Employees per Location [employee_location_ratio]", "Government / National Lab [gov_lab_flag]"])
df_compressed.info()

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = [
    "Employee Count [employee_count]",
    "Revenue [revenue]",
    "Year Founded [year_founded]",
    "Number of Locations [number_of_locations]",
]

cat_cols = [
    "Industry [industry]",
    "HQ State / Region [hq_region]",
    "Ownership Type [public_private]",
]

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='constant', fill_value="Unknown")

df_compressed[num_cols] = num_imputer.fit_transform(df_compressed[num_cols])
df_compressed[cat_cols] = cat_imputer.fit_transform(df_compressed[cat_cols])

df_compressed.info()

In [ ]:
tech_cols = ["Total Tech Tools [total_tech_count]", "Tech Complexity Score [tech_stack_complexity_score]"]
tech_scaler = StandardScaler()
tech_pca = PCA(n_components=1)

tech_scaled_train = tech_scaler.fit_transform(df_compressed[tech_cols].fillna(0))
df_compressed['PCA_Tech_Maturity'] = tech_pca.fit_transform(tech_scaled_train)
df_compressed.info()
print(f"Variance captured by PCA: {tech_pca.explained_variance_ratio_[0]:.2%}")

In [ ]:
df_compressed.head()

In [ ]:
nlp_model = SentenceTransformer('all-MiniLM-L6-v2')
df_industries = df_compressed['Industry [industry]'].tolist()
df_embeddings = nlp_model.encode(df_industries)

industry_pca = PCA(n_components=5)
df_industries_vecs = industry_pca.fit_transform(df_embeddings)

for i in range(5):
  df_compressed[f'Industry_Vec_{i}'] = df_industries_vecs[:, i]

df_compressed.info()

In [ ]:
df_region_coordinates = pd.read_csv("US_GeoCode.csv")
df_region_coordinates.head()

In [ ]:
region_coordinates = dict(
    zip(
        df_region_coordinates['Name'],
        zip(df_region_coordinates['latitude'], df_region_coordinates['longitude'])
    )
)

region_coordinates['Unknown'] = (0.0,0.0)
print("dictionary sample: ", list(region_coordinates.items())[:5])

In [ ]:
def get_lat_lon(state_name):
  return region_coordinates.get(state_name, (0.0,0.0))

train_coords = df_compressed['HQ State / Region [hq_region]'].apply(get_lat_lon)
df_compressed["HQ_Lat"] = train_coords.apply(lambda x: x[0])
df_compressed["HQ_Lon"] = train_coords.apply(lambda x: x[1])

df_compressed.info()

In [ ]:
df_compressed.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

mapping_dict = {"Yes": 1, "No": 0}
df_compressed["Gov_Lab_Numeric"] = df_compressed["Government / National Lab [gov_lab_flag]"].map(mapping_dict).fillna(0)

df_compressed["Ownership Type [public_private]"] = df_compressed["Ownership Type [public_private]"].str.strip().str.lower()

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_train_array = encoder.fit_transform(df_compressed[["Ownership Type [public_private]"]])
encoded_col_names = encoder.get_feature_names_out(["Ownership Type [public_private]"])

df_encoded = pd.DataFrame(encoded_train_array, columns=encoded_col_names, index=df_compressed.index)
df_compressed = pd.concat([df_compressed, df_encoded], axis=1)

df_compressed.info()

In [ ]:
df_compressed = df_compressed.drop(df_compressed.columns[[23,24,25,26,27,28,29]], axis=1)
df_compressed.info()

In [ ]:
df_compressed.info()

In [ ]:
final_features = [
    "Employee Count [employee_count]",
    "Revenue [revenue]",
    "Year Founded [year_founded]",
    "Number of Locations [number_of_locations]",
    "Employees per Location [employee_location_ratio]",
    "Infrastructure Maturity Score [infrastructure_maturity]",
    "Endpoint Density [endpoint_density_proxy]",
    "PCA_Tech_Maturity",
    "Industry_Vec_0",
    "Industry_Vec_1",
    "Industry_Vec_2",
    "Industry_Vec_3",
    "Industry_Vec_4",
    "HQ_Lat",
    "HQ_Lon",
    "Gov_Lab_Numeric",
    "Ownership Type [public_private]_private",
    "Ownership Type [public_private]_public",
    "Ownership Type [public_private]_unknown"
]

master_scaler = StandardScaler()
X_train_scaled = master_scaler.fit_transform(df_compressed[final_features])

inertia = []
sil_scores = []
K_range = range(2,8)

for k in K_range:
  kmeans_eval = KMeans(n_clusters=k, random_state=42, n_init=10)
  kmeans_eval.fit(X_train_scaled)

  inertia.append(kmeans_eval.inertia_)
  sil_scores.append(silhouette_score(X_train_scaled, kmeans_eval.labels_))

fig, ax = plt.subplots(1,2, figsize=(12,4))

# Chart 1: The Elbow
ax[0].plot(K_range, inertia, marker='o')
ax[0].set_title('Elbow Method (Look for the bend)')
ax[0].set_xlabel('Number of Clusters (K)')
ax[0].set_ylabel('Inertia')

# Chart 2: The Silhouette
ax[1].plot(K_range, sil_scores, marker='o', color='orange')
ax[1].set_title('Silhouette Score (Look for the peak)')
ax[1].set_xlabel('Number of Clusters (K)')
ax[1].set_ylabel('Score')

plt.show()

In [ ]:
OPTIMAL_K = 4

final_kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df_compressed["Cluster_ID"] = final_kmeans.fit_predict(X_train_scaled)
df_compressed.head()

In [ ]:
# Grab just the Name and the Cluster ID, and sort them by Cluster
company_list = df_compressed[['Cluster_ID', 'Company Name [canonical_name]', 'Revenue [revenue]', 'Industry [industry]']].copy()
company_list = company_list.sort_values(by='Cluster_ID').reset_index(drop=True)

# Rename columns for presentation
company_list.columns = ['Cluster ID', 'Company', 'Revenue', 'Industry']

# Display the first 50 rows
display(company_list.head(126))
company_list.to_csv('amd_focus_100_clusters.csv', index=False)

In [ ]:
# 1. Compress the 14D scaled data down to 2D for visualization
pca_2d = PCA(n_components=2)
X_train_2d = pca_2d.fit_transform(X_train_scaled)

# Squash the centroids down to the exact same 2D map
centroids_2d = pca_2d.transform(final_kmeans.cluster_centers_)

# 2. Add the new X/Y coordinates back to your dataframe for plotting
df_compressed['Plot_X'] = X_train_2d[:, 0]
df_compressed['Plot_Y'] = X_train_2d[:, 1]

# 3. Set up the visual canvas
plt.figure(figsize=(14, 10))
sns.set_theme(style="whitegrid")

# 4. Plot all 100 Companies as colored dots
sns.scatterplot(
    data=df_compressed,
    x='Plot_X',
    y='Plot_Y',
    hue='Cluster_ID',
    palette='Set2',  # Distinct colors for each persona
    alpha=0.8,
    s=120,           # Size of the company dots
    edgecolor='black'
)

# 5. Plot the 4 Centroids as giant red stars
plt.scatter(
    centroids_2d[:, 0],
    centroids_2d[:, 1],
    marker='*',
    s=1000,
    c='red',
    edgecolor='black',
    label='Persona Centroid'
)

# 6. Draw lines connecting each company to its Centroid to show "Proximity"
for i in range(len(df_compressed)):
    # Get the cluster assignment and coordinates for this specific company
    cluster_idx = df_compressed['Cluster_ID'].iloc[i]
    company_coords = (df_compressed['Plot_X'].iloc[i], df_compressed['Plot_Y'].iloc[i])
    centroid_coords = (centroids_2d[cluster_idx, 0], centroids_2d[cluster_idx, 1])

    # Draw a faint line between the company and its centroid
    plt.plot(
        [company_coords[0], centroid_coords[0]],
        [company_coords[1], centroid_coords[1]],
        c='gray',
        alpha=0.2,
        linewidth=1,
        zorder=0  # Keeps lines behind the dots
    )

# 7. Add titles and labels
plt.title('AMD Target Personas: K-Means Clustering (K=4)\nVisualizing Proximity to Centroid', fontsize=16, pad=20)
plt.xlabel(f'Principal Component 1 ({pca_2d.explained_variance_ratio_[0]:.1%} Variance Explained)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({pca_2d.explained_variance_ratio_[1]:.1%} Variance Explained)', fontsize=12)

# Move the legend outside the chart so it doesn't cover data points
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
import joblib
import os

# Create the models folder if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save all your trained tools to the hard drive
joblib.dump(num_imputer, 'models/num_imputer.pkl')
joblib.dump(cat_imputer, 'models/cat_imputer.pkl')
joblib.dump(tech_scaler, 'models/tech_scaler.pkl')
joblib.dump(tech_pca, 'models/tech_pca.pkl')
joblib.dump(industry_pca, 'models/industry_pca.pkl')
joblib.dump(encoder, 'models/ownership_encoder.pkl')
joblib.dump(master_scaler, 'models/master_scaler.pkl')
joblib.dump(final_kmeans, 'models/kmeans_model.pkl')

print("Pipeline artifacts successfully saved to /models!")